---
### **Robustness Test**

강건성 검사
- 분위수 검증
- 5x5 Double Sort
- 민감도 분석
- 스프레드 회귀

---
## **0. 포트폴리오 계산**

##### **데이터 로드**

In [42]:
import pandas as pd
import numpy as np
from module_quarter import performance_metrics

total_adj_close     = pd.read_csv('../../00_input/KOSPI_KOSDAQ_제조업_total_adj_close.csv', index_col='Date', parse_dates=True)
trading_value_60    = pd.read_csv('../../00_input/KOSPI_KOSDAQ_제조업_trading_value_60.csv', index_col='Date', parse_dates=True)
trading_value       = pd.read_csv('../../00_input/KOSPI_KOSDAQ_제조업_trading_value.csv', index_col='Date', parse_dates=True)
mkt_cap             = pd.read_csv('../../00_input/KOSPI_KOSDAQ_제조업_mkt_cap.csv', index_col='Date', parse_dates=True)
adj_close           = pd.read_csv('../../00_input/KOSPI_KOSDAQ_제조업_adj_close.csv', index_col='Date', parse_dates=True)
factor_df           = pd.read_csv("../../00_input/profitability_제조업_분기_v3.csv", index_col=0, parse_dates=True) 

##### **포트폴리오 함수 정의**

In [43]:
from joblib import Parallel, delayed
from tqdm import tqdm

def run_strategy(
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp, n_temp,
        monthly_rets, illiq,
        sort_mode  = 'single',  # 'single', 'double', 'spread'
        c_temp     = None,      # Double Sort 시가총액 분위수
        spread_leg = None       # Spread 회귀 ('long' or 'short')
):
    
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    # 포트폴리오 이름 생성
    if sort_mode == 'single':
        portfolio_name = f"{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_Q({n_temp+1}/{q_cut})"
    elif sort_mode == 'double': # For Double Sorting
        portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({n_temp+1}/{q_cut})"
    elif sort_mode == 'spread': # For Spread Regression
        portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({spread_leg})"

    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    monthly_rets_win = monthly_rets.clip(
        lower=monthly_rets.quantile(wins_threshold_temp),
        upper=monthly_rets.quantile(1 - wins_threshold_temp),
        axis = 1
    )

    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 시가총액 및 유동성 필터링
        if sort_mode in ['double', 'spread']:
            cap_series      = mkt_cap.loc[start_date]
            cap_quantile    = pd.qcut(cap_series, q=cap_cut, labels=False)
            filtered_index  = cap_series[cap_quantile == c_temp].index
            series          = trading_value_60.loc[start_date, filtered_index]
        else:
            series          = trading_value_60.loc[start_date]

        threshold       = series.quantile(trading_threshold_temp)
        filtered        = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index].dropna()

        # 종목 선정 (분위수 or 롱숏)
        if sort_mode == 'spread':
            low_th  = factor_filtered.quantile(0.3)
            high_th = factor_filtered.quantile(0.7)
            if spread_leg == 'long':
                basket = factor_filtered[factor_filtered <= low_th].index
            else:
                basket = factor_filtered[factor_filtered > high_th].index
        else:
            quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
            basket    = factor_filtered[quantiles == n_temp].index

        # Amihud High Cost 종목 선정
        illiq_startdate = illiq.loc[start_date].dropna()
        threshold       = illiq_startdate.quantile(1-Amihud_threshold_temp)
        illiq_top       = illiq_startdate[illiq_startdate >= threshold].index

        # 비중 할당
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1.0/len(basket), index=basket)
        else: # 'Cap'
            cap_seg        = mkt_cap.loc[start_date, basket]
            target_weights = cap_seg / cap_seg.sum()

        # 거래 비용 및 NAV 업데이트
        all_index = target_weights.index.union(prev_weights.index)
        target_w  = target_weights.reindex(all_index, fill_value=0)
        prev_w    = prev_weights.reindex(all_index, fill_value=0)
        delta_w   = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiq_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()

        NAV_new       = NAV - trade_cost

        current_portfolio_value = target_weights * NAV_new
        ret_seg                 = monthly_rets_win.loc[end_date, basket]
        next_portfolio_value    = current_portfolio_value * (ret_seg + 1)

        NAV_new                 = next_portfolio_value.sum()
        portfolio_ret           = NAV_new / NAV - 1

        prev_portfolio          = next_portfolio_value
        NAV                     = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    # 결과 반환
    return {'return': portfolio_return, 'trade': total_trade}

In [44]:
def select_columns(df, fixed_options):
    # {기존 이름: 새 이름} 딕셔너리 생성
    rename_dict = {
        c: "_".join([c.split('_')[i] for i, v in fixed_options.items() if v == "*"])
        for c in df.columns if all(c.split('_')[i] == v or v == "*" for i, v in fixed_options.items())
    }
    
    # 컬럼 선택 후 반환
    return df[rename_dict.keys()].rename(columns=rename_dict)

---
##### **설정**

In [45]:
start_point       = '2006-03-31'                     # 백테스트 기간 설정
end_point         = '2025-12-31'

q_cut             = 5                                # 포트폴리오를 나누는 분위 수    e.g. 5 → 5분위 
n                 = [0, 1, 2, 3, 4]                  # 0 : 하위 ~ (q_cut - 1) : 상위  e.g. [0, 1, 2, 3, 4]

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']
cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)         e.g. [(0.008, 0.003), (0, 0)]
wins_threshold    = [0.01]                           # 수익률 윈저라이징               e.g. [0.01, 0.05]
trading_threshold = [0.1]                            # 60일 평균 거래대금 필터링        e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]                            # 유동성 기준                    e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

##### **계산 실행**

In [46]:
# 일별 수익률과 거래대금을 이용해 Amihud illiquidity 계산
daily_ret        = adj_close.pct_change(fill_method=None)
daily_illiq      = (daily_ret.abs()/trading_value)
illiq            = daily_illiq.resample('QE').mean()

In [47]:
# monthly_rets        = total_adj_close.resample('QE').pct_change(fill_method=None) # 분기 리밸런싱
monthly_rets        = total_adj_close.resample('QE').last().pct_change(fill_method=None)

portfolio_df = pd.DataFrame()
trade_df     = pd.DataFrame()

month_ends  = pd.date_range(start=start_point, end=end_point, freq='QE')

In [48]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(*params, monthly_rets, illiq) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df = pd.DataFrame({r['return'].name: r['return'] for r in results})
trade_df     = pd.DataFrame({r['trade'].name: r['trade'] for r in results})

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:05<00:00,  3.62it/s]


5분위 포트폴리오 저장

In [49]:
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")
# 인덱스(날짜) 포함, Excel에서 한글 깨짐 방지하려면 encoding='utf-8-sig'
portfolio_df.to_csv(out_dir / "5분위_portfolio_return_all.csv", index_label="Date", encoding="utf-8-sig")
trade_df.to_csv(out_dir / "5분위_total_trade_all.csv", index_label="Date", encoding="utf-8-sig")

##### **Output**

In [50]:
portfolio_df.tail()

,Equal_R0.01_H80L30_T0.1_A0.2_Q(1/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(2/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(3/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(4/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(5/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(1/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(2/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(3/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(4/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(5/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(1/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(2/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(3/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(4/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(5/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(1/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(2/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(3/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(4/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(5/5)
2024-12-31,-0.039745,-0.062430,-0.085650,-0.074102,-0.125269,-0.038506,-0.060542,-0.083288,-0.071796,-0.123574,-0.019427,-0.105423,-0.097931,-0.042634,-0.210031,-0.018112,-0.104500,-0.097422,-0.039718,-0.207496
2025-03-31,0.011595,0.019972,-0.004568,-0.014184,-0.044859,0.012864,0.022135,-0.002074,-0.011777,-0.043038,0.050503,0.071307,0.010651,0.034048,-0.001442,0.053424,0.075287,0.012721,0.036876,-0.000255
2025-06-30,0.190945,0.183762,0.200215,0.180315,0.126945,0.193493,0.188250,0.205754,0.185703,0.130556,0.187176,0.284066,0.283691,0.186139,0.119221,0.188941,0.286530,0.288929,0.191303,0.122094
2025-09-30,0.030658,0.030146,0.014154,0.012823,0.027404,0.031755,0.032021,0.016427,0.015178,0.029318,0.173629,0.096337,0.030230,0.092418,0.035013,0.173759,0.097679,0.031723,0.093069,0.035404
2025-12-31,0.003139,0.010870,0.041260,0.047097,0.039803,0.004322,0.012978,0.043946,0.049801,0.041782,0.362769,0.037441,0.166361,0.158532,0.173916,0.362922,0.038123,0.167094,0.158995,0.174696


---
## **1. 분위수 검증**

##### **포트폴리오 선택**

In [51]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

In [52]:
subset.tail()

,Q(1/5),Q(2/5),Q(3/5),Q(4/5),Q(5/5)
2024-12-31,-0.019427,-0.105423,-0.097931,-0.042634,-0.210031
2025-03-31,0.050503,0.071307,0.010651,0.034048,-0.001442
2025-06-30,0.187176,0.284066,0.283691,0.186139,0.119221
2025-09-30,0.173629,0.096337,0.030230,0.092418,0.035013
2025-12-31,0.362769,0.037441,0.166361,0.158532,0.173916


In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav, log_cum = (subset + 1).cumprod(), np.log1p(subset).cumsum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 단일 루프로 두 그래프 그리기
for col in subset.columns:
    fig.add_trace(go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col, legendgroup=col), row=1, col=1)
    fig.add_trace(go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col, legendgroup=col), row=1, col=2)

# 레이아웃 및 테두리 설정
fig.update_layout(title="Portfolio Performance Comparison", height=500, width=1000, template="plotly_white")
fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 x축 테두리
fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 y축 테두리

fig.show()

In [54]:
import numpy as np
import pandas as pd
import plotly.express as px

# run_strategy와 동일한 규칙으로 컬럼명 생성
q_cut = 5
wins_threshold_temp = 0.01
weight_method_temp = "Cap"
trading_threshold_temp = 0.1   # 거래대금 하위 10% 제외(상위 90%)
Amihud_threshold_temp = 0.2
n_temp = 0

def portfolio_col_name(high_cost, low_cost):
    return (
        f"{weight_method_temp}_R{wins_threshold_temp}_"
        f"H{int(high_cost * 10000)}L{int(low_cost * 10000)}_"
        f"T{trading_threshold_temp}_A{Amihud_threshold_temp}_"
        f"Q({n_temp + 1}/{q_cut})"
    )

col_h80l30 = portfolio_col_name(0.008, 0.003)
col_h0l0 = portfolio_col_name(0.0, 0.0)

subset_cost = pd.DataFrame(
    {
        "H80L30": portfolio_df[col_h80l30],
        "H0L0": portfolio_df[col_h0l0],
    }
)

log_cum_cost = np.log1p(subset_cost).cumsum()

fig = px.line(
    log_cum_cost,
    title="Cap-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    labels={"value": "Log Cumulative Return", "index": "Date", "variable": "Cost Parameter"},
    template="plotly_white",
    width=900,
    height=600,
)

fig.update_xaxes(showline=True, linewidth=1, linecolor="gray", mirror=True)
fig.update_yaxes(showline=True, linewidth=1, linecolor="gray", mirror=True)

fig.show()

---
## **2. Double sort : Size Control**


##### **계산 실행**

동일가중

In [55]:
cap_cut           = 5                          # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
c                 = [0, 1, 2, 3, 4]    

In [56]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[0]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
trading_threshold_temp = trading_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [57]:
# --- 병렬 실행 ---
results_double_equal = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        wins_threshold_temp, weight_method_temp, cost_temp, 
        trading_threshold_temp, Amihud_threshold_temp, n_temp,
        monthly_rets, illiq,    
        sort_mode='double',      # 'double'로 변경
        c_temp=c_temp            # 시가총액 분위수 추가
    )
    for c_temp in c
    for n_temp in n
)

# --- 결과 합치기 ---
cap_cut_portfolio_equal_df = pd.DataFrame({r['return'].name: r['return'] for r in results_double_equal})

25가지 포트폴리오 저장

In [58]:
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")

cap_cut_trade_equal_df = pd.DataFrame({r["trade"].name: r["trade"] for r in results_double_equal})

cap_cut_portfolio_equal_df.to_csv(
    out_dir / "double_sort_portfolio_equal_return_25.csv",
    index_label="Date",
    encoding="utf-8-sig",
)
cap_cut_trade_equal_df.to_csv(
    out_dir / "double_sort_total_trade_equal_25.csv",
    index_label="Date",
    encoding="utf-8-sig",
)

##### **Output**

In [59]:
cap_cut_portfolio_equal_df.head()

,C(1/5)_Q(1/5),C(1/5)_Q(2/5),C(1/5)_Q(3/5),C(1/5)_Q(4/5),C(1/5)_Q(5/5),C(2/5)_Q(1/5),C(2/5)_Q(2/5),C(2/5)_Q(3/5),C(2/5)_Q(4/5),C(2/5)_Q(5/5),...,C(4/5)_Q(1/5),C(4/5)_Q(2/5),C(4/5)_Q(3/5),C(4/5)_Q(4/5),C(4/5)_Q(5/5),C(5/5)_Q(1/5),C(5/5)_Q(2/5),C(5/5)_Q(3/5),C(5/5)_Q(4/5),C(5/5)_Q(5/5)
2006-06-30,-0.053951,-0.059890,0.024188,-0.060300,0.043863,-0.133987,-0.091716,-0.105694,-0.132429,-0.091415,...,-0.116318,-0.131395,-0.146057,-0.143577,-0.200995,-0.102169,-0.034602,-0.078434,-0.140238,-0.154045
2006-09-30,0.085973,0.152004,0.195319,0.101875,0.043849,0.048483,0.061233,0.056653,-0.023551,0.030672,...,0.081091,0.038673,0.037924,0.065379,-0.020653,0.017962,0.074872,0.054532,0.097290,0.096911
2006-12-31,0.103616,0.234473,0.138645,0.152580,-0.034010,0.097818,0.013971,0.050137,0.053225,-0.172370,...,0.051936,0.029869,0.077089,-0.051680,-0.010859,0.084822,0.065841,0.100463,0.068883,0.050102
2007-03-31,0.357396,0.207358,0.298975,0.281389,0.218478,0.195390,0.129202,0.112119,0.083752,0.158274,...,-0.005944,0.063409,0.007249,0.028280,-0.036881,-0.011686,-0.008714,0.023811,-0.016711,0.008365
2007-06-30,0.267637,0.187596,0.243968,0.158432,0.162728,0.232232,0.316049,0.218231,0.213727,0.281753,...,0.276760,0.295322,0.272847,0.242801,0.127150,0.186714,0.263790,0.391648,0.350964,0.214214


---
##### **Plot**

##### **Code**

In [60]:
# 월별 수익률 시리즈 기준 CAGR 계산
def CAGR(series):
    s = series.dropna()
    if s.empty: return np.nan
    return (1 + s).prod() ** (365.25 / (s.index[-1] - s.index[0]).days) - 1

In [61]:
# 2. Heatmap 매트릭스 생성
cagrs = cap_cut_portfolio_equal_df.apply(CAGR)
cagrs.index = cagrs.index.str.split('_', expand=True)
cagr_matrix = cagrs.unstack()

In [62]:
import plotly.express as px

# 3. 시각화
fig = px.imshow(
    cagr_matrix, text_auto=".1%", color_continuous_scale="Blues", aspect="equal",
    title="CAGR Heatmap by Cap vs Factor Quintile", width=600, height=600,
    labels=dict(x="Factor Quintile (Q)", y="Market Cap Quintile (C)", color="CAGR")
)

fig.show()

시가총액

In [63]:
cap_cut           = 5                          # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
c                 = [0, 1, 2, 3, 4]    

In [64]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[1]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
trading_threshold_temp = trading_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [65]:
# --- 병렬 실행 ---
results_double_cap = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        wins_threshold_temp, weight_method_temp, cost_temp, 
        trading_threshold_temp, Amihud_threshold_temp, n_temp,
        monthly_rets, illiq,    
        sort_mode='double',      # 'double'로 변경
        c_temp=c_temp            # 시가총액 분위수 추가
    )
    for c_temp in c
    for n_temp in n
)

# --- 결과 합치기 ---
cap_cut_portfolio_cap_df = pd.DataFrame({r['return'].name: r['return'] for r in results_double_cap})

In [66]:
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")

cap_cut_trade_cap_df = pd.DataFrame({r["trade"].name: r["trade"] for r in results_double_cap})

cap_cut_portfolio_cap_df.to_csv(
    out_dir / "double_sort_portfolio_cap_return_25.csv",
    index_label="Date",
    encoding="utf-8-sig",
)
cap_cut_trade_cap_df.to_csv(
    out_dir / "double_sort_total_trade_cap_25.csv",
    index_label="Date",
    encoding="utf-8-sig",
)

In [67]:
cap_cut_portfolio_cap_df.head()

,C(1/5)_Q(1/5),C(1/5)_Q(2/5),C(1/5)_Q(3/5),C(1/5)_Q(4/5),C(1/5)_Q(5/5),C(2/5)_Q(1/5),C(2/5)_Q(2/5),C(2/5)_Q(3/5),C(2/5)_Q(4/5),C(2/5)_Q(5/5),...,C(4/5)_Q(1/5),C(4/5)_Q(2/5),C(4/5)_Q(3/5),C(4/5)_Q(4/5),C(4/5)_Q(5/5),C(5/5)_Q(1/5),C(5/5)_Q(2/5),C(5/5)_Q(3/5),C(5/5)_Q(4/5),C(5/5)_Q(5/5)
2006-06-30,-0.052934,-0.064367,0.018374,-0.103250,0.023038,-0.137003,-0.094544,-0.112671,-0.129961,-0.082231,...,-0.112083,-0.115474,-0.140882,-0.140064,-0.208320,-0.034750,-0.021947,-0.045089,-0.136941,-0.058377
2006-09-30,0.079802,0.146740,0.207540,0.083687,0.034651,0.049965,0.062869,0.053866,-0.020853,0.016634,...,0.082077,0.040490,0.037281,0.060745,-0.014351,0.060537,0.033307,0.067700,0.042841,0.109186
2006-12-31,0.099562,0.240695,0.105822,0.144349,-0.043673,0.097623,0.005889,0.051728,0.050828,-0.159311,...,0.059735,0.033621,0.073903,-0.043437,-0.010788,0.034519,0.074393,0.047286,0.030021,0.065357
2007-03-31,0.364034,0.199740,0.305279,0.221023,0.195240,0.197153,0.127097,0.112794,0.083555,0.150804,...,-0.010276,0.055157,0.006715,0.035811,-0.035175,-0.025775,-0.027497,0.023160,-0.006352,0.075127
2007-06-30,0.266392,0.183682,0.252883,0.171869,0.205787,0.236270,0.322831,0.221387,0.210040,0.293926,...,0.255023,0.308983,0.287877,0.244555,0.140244,0.075669,0.183861,0.330180,0.420861,0.269907


In [68]:
# 2. Heatmap 매트릭스 생성
cagrs = cap_cut_portfolio_cap_df.apply(CAGR)
cagrs.index = cagrs.index.str.split('_', expand=True)
cagr_matrix = cagrs.unstack()

In [69]:
import plotly.express as px

# 3. 시각화
fig = px.imshow(
    cagr_matrix * 100, text_auto=".1f", color_continuous_scale="Blues", aspect="equal",
    title="CAGR Heatmap by Cap vs Factor Quintile", width=600, height=600,
    labels=dict(x="Factor Quintile (Q)", y="Market Cap Quintile (C)", color="CAGR")
)
fig.update_traces(textfont=dict(size=25))
fig.show()

---
## **3. Parameter Sensitivity**
`0. 포트폴리오 계산`에서 계산한 포트폴리오 `portfolio_df` 데이터 사용

---
#### **1) Weight sensitivity**

##### **포트폴리오 선택**

In [70]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_equal = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

# 시총가중 포트폴리오
subset_cap = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

##### **Code**

In [71]:
# 데이터 준비
log_cum_eq, log_cum_cap = np.log1p(subset_equal).cumsum(), np.log1p(subset_cap).cumsum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Equal-weighted Portfolios", "Cap-weighted Portfolios"])

# 단일 루프로 두 그래프 동시에 그리기
for col in log_cum_eq.columns:
    # 1열: Equal 가중 그래프
    fig.add_trace(go.Scatter(x=log_cum_eq.index, y=log_cum_eq[col], mode='lines', name=f"Equal_{col}", legendgroup=col), row=1, col=1)
    # 2열: Cap 가중 그래프
    fig.add_trace(go.Scatter(x=log_cum_cap.index, y=log_cum_cap[col], mode='lines', name=f"Cap_{col}", legendgroup=col), row=1, col=2)

# 레이아웃 및 테두리 설정
fig.update_layout(title="Equal vs Cap Weighted Portfolios (Log Cumulative Returns)", height=500, width=1000, template="plotly_white")
fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 x축 테두리
fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 y축 테두리

fig.show()

---
#### **2) Cost sensitivity**

##### **포트폴리오 선택**

In [72]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_cost = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "*", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
)

##### **Code**

In [73]:
# 데이터 준비
log_cum_cost = np.log1p(subset_cost).cumsum()

# DataFrame 렌더링 & 레이아웃 설정
fig = px.line(
    log_cum_cost, 
    title="Cap-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    labels={"value": "Log Cumulative Return", "index": "Date", "variable": "Cost Parameter"},
    template="plotly_white", width=900, height=600
)

# 스타일 통일 적용
fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)
fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)

fig.show()

---
## **4. 스프레드 회귀 (Factor regression)**

##### **데이터 로드**

In [74]:
factors        = pd.read_csv("../../00_input/factors quarterly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)
portfolio      = portfolio_df['Cap_R0.01_H80L30_T0.1_A0.2_Q(1/5)']
portfolio.name = 'Return'

##### **계산 실행**

In [75]:
cap_cut        = 2                  # 포트폴리오를 나누는 분위 수 e.g. 2 → 2분위 
c              = [0, 1]    
spread_legs    = ['long', 'short']

In [76]:
# 단일 병렬 실행
results_spread = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        wins_threshold[0], weight_method[0], cost[0], 
        trading_threshold[0], Amihud_threshold[0], None, # spread 모드에서는 n_temp가 무시되므로 None
        monthly_rets, illiq,                        
        sort_mode='spread', c_temp=c_temp, spread_leg=spread_leg
    )
    for c_temp in c
    for spread_leg in spread_legs
)

factor_portfolio_df = pd.DataFrame({r['return'].name: r['return'] for r in results_spread})

In [77]:
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")

factor_trade_df = pd.DataFrame({r["trade"].name: r["trade"] for r in results_spread})

factor_portfolio_df.to_csv(
    out_dir / "spread_portfolio_return.csv",
    index_label="Date",
    encoding="utf-8-sig",
)
factor_trade_df.to_csv(
    out_dir / "spread_total_trade.csv",
    index_label="Date",
    encoding="utf-8-sig",
)

In [78]:
factor_portfolio_df.tail()

,C(1/2)_Q(long),C(1/2)_Q(short),C(2/2)_Q(long),C(2/2)_Q(short)
2024-12-31,-0.038276,-0.111573,-0.045149,-0.109749
2025-03-31,0.000418,-0.050247,0.023361,-0.000323
2025-06-30,0.161287,0.110551,0.206838,0.189063
2025-09-30,0.007300,0.005108,0.033419,0.047620
2025-12-31,-0.012810,-0.016190,0.013704,0.119907


In [79]:
# long-short 스프레드 계산
small_ls = factor_portfolio_df["C(1/2)_Q(long)"] - factor_portfolio_df["C(1/2)_Q(short)"]
big_ls   = factor_portfolio_df["C(2/2)_Q(long)"] - factor_portfolio_df["C(2/2)_Q(short)"]

# 두 그룹 평균 → 최종 factor return
factor_return = (small_ls + big_ls) / 2
factor_return.name = "Factor"

##### **Output : Profitability 팩터 포트폴리오** 

In [80]:
factor_return.head()

2006-06-30    0.022032
2006-09-30    0.038775
2006-12-31    0.116155
2007-03-31    0.005430
2007-06-30    0.070200
Name: Factor, dtype: float64

##### **회귀분석**

##### **Code**

In [81]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

In [82]:
import statsmodels.api as sm

# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

##### **기존 결과**

In [83]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.875
Method:                 Least Squares   F-statistic:                     441.5
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           8.40e-51
Time:                        15:36:31   Log-Likelihood:                 160.73
No. Observations:                  79   AIC:                            -311.5
Df Residuals:                      74   BIC:                            -299.6
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0183      0.004      5.204      0.0

---

##### **Code**

In [84]:
# factor_return 추가
df["Factor"] = factor_return.loc[df.index]

# 독립변수: MKT, SMB, HML, MOM, MyFactor
X_ext = X.copy()
X_ext["Factor"] = df["Factor"]

X2 = sm.add_constant(X_ext)

# 4. OLS 회귀
model = sm.OLS(y, X2).fit(cov_type="HAC", cov_kwds={"maxlags":12})

##### **새로운 팩터를 포함한 결과**

In [85]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.884
Model:                            OLS   Adj. R-squared:                  0.876
Method:                 Least Squares   F-statistic:                     410.2
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           6.14e-52
Time:                        15:36:31   Log-Likelihood:                 161.60
No. Observations:                  79   AIC:                            -311.2
Df Residuals:                      73   BIC:                            -297.0
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0148      0.005      2.883      0.0

---
## **5. LongShort Test**

동일가중

In [86]:
def run_strategy_spread_qcut(
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp,
        monthly_rets, illiq,
        c_temp=None,               # double sort의 cap bucket
        long_bucket='Q1',          # 'Q1' or 'Q5'
        q_cut_local=5,             # 분위수
        sort_mode='spread'         # 기존 호환
):
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({long_bucket})" if c_temp is not None else f"Q({long_bucket})"
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade = pd.Series(dtype=float, name=portfolio_name)

    monthly_rets_win = monthly_rets.clip(
        lower=monthly_rets.quantile(wins_threshold_temp),
        upper=monthly_rets.quantile(1 - wins_threshold_temp),
        axis=1
    )

    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 1) 필터링
        if c_temp is not None:
            cap_series = mkt_cap.loc[start_date].dropna()
            cap_q = pd.qcut(cap_series, q=cap_cut, labels=False, duplicates='drop')
            filtered_index = cap_series[cap_q == c_temp].index
            tv = trading_value_60.loc[start_date, filtered_index].dropna()
        else:
            tv = trading_value_60.loc[start_date].dropna()

        tv_th = tv.quantile(trading_threshold_temp)
        filtered = tv[tv > tv_th].index

        fac = factor_df.loc[start_date, filtered].dropna()
        if len(fac) < q_cut_local:
            continue

        # 2) 5분위 기준 basket 선택
        fac_q = pd.qcut(fac, q=q_cut_local, labels=False, duplicates='drop')
        if long_bucket == 'Q1':
            target_label = fac_q.min()     # 보통 0
        elif long_bucket == 'Q5':
            target_label = fac_q.max()     # 보통 4
        else:
            raise ValueError("long_bucket은 'Q1' 또는 'Q5'만 허용")

        basket = fac[fac_q == target_label].index
        if len(basket) == 0:
            continue

        # 3) 거래비용/비중
        illiq_start = illiq.loc[start_date].dropna()
        illiq_th = illiq_start.quantile(1 - Amihud_threshold_temp)
        illiq_top = illiq_start[illiq_start >= illiq_th].index

        if prev_portfolio is None:
            prev_weights = pd.Series(0.0, index=basket)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1.0 / len(basket), index=basket)
        else:
            cap_seg = mkt_cap.loc[start_date, basket]
            target_weights = cap_seg / cap_seg.sum()

        all_index = target_weights.index.union(prev_weights.index)
        target_w = target_weights.reindex(all_index, fill_value=0.0)
        prev_w = prev_weights.reindex(all_index, fill_value=0.0)
        delta_w = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate = pd.Series(low_cost, index=all_index)
        cost_rate.loc[cost_rate.index.intersection(illiq_top)] = high_cost
        trade_cost = (trade_amounts * cost_rate).sum()

        NAV_after_cost = NAV - trade_cost
        cur_val = target_weights * NAV_after_cost
        ret_seg = monthly_rets_win.loc[end_date, basket]
        next_val = cur_val * (1 + ret_seg)

        NAV_new = next_val.sum()
        port_ret = NAV_new / NAV - 1

        prev_portfolio = next_val
        NAV = NAV_new

        total_trade.loc[start_date] = trade_amounts.sum()
        portfolio_return.loc[end_date] = port_ret

    return {'return': portfolio_return, 'trade': total_trade}

In [87]:
from joblib import Parallel, delayed

cap_cut = 5
cap_buckets = list(range(cap_cut))  # 0~4

# Q1 leg
res_q1 = Parallel(n_jobs=-1)(
    delayed(run_strategy_spread_qcut)(
        wins_threshold[0], weight_method[0], cost[0],
        trading_threshold[0], Amihud_threshold[0],
        monthly_rets, illiq,
        c_temp=c, long_bucket='Q1', q_cut_local=5
    )
    for c in cap_buckets
)

# Q5 leg
res_q5 = Parallel(n_jobs=-1)(
    delayed(run_strategy_spread_qcut)(
        wins_threshold[0], weight_method[0], cost[0],
        trading_threshold[0], Amihud_threshold[0],
        monthly_rets, illiq,
        c_temp=c, long_bucket='Q5', q_cut_local=5
    )
    for c in cap_buckets
)

q1_df = pd.DataFrame({r['return'].name: r['return'] for r in res_q1})
q5_df = pd.DataFrame({r['return'].name: r['return'] for r in res_q5})

# c_temp별 롱숏: Q1 - Q5
ls_df = pd.DataFrame(index=q1_df.index)
for i in range(cap_cut):
    c = f"C({i+1}/{cap_cut})"
    ls_df[f"{c}_LS(Q1-Q5)"] = q1_df[f"{c}_Q(Q1)"] - q5_df[f"{c}_Q(Q5)"]

# 전체 평균 롱숏 팩터
factor_return = ls_df.mean(axis=1)
factor_return.name = "Factor_LS_Q1_minus_Q5"
factor_return.head()

2006-06-30    0.008467
2006-09-30    0.061029
2006-12-31    0.135661
2007-03-31    0.049801
2007-06-30    0.078059
Name: Factor_LS_Q1_minus_Q5, dtype: float64

In [88]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 포트폴리오/팩터 수익률 시계열 (직전 코드에서 만든 factor_return을 그대로 사용)
portfolio = factor_return
port_name = portfolio.name if portfolio.name is not None else "Factor_LS(Q1-Q5)"

# 데이터 준비
nav = (1 + portfolio).cumprod()                 # 누적 NAV
log_cum = np.log1p(portfolio).cumsum()        # 로그 누적 수익률

# 서브플롯 (가로 2개)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Cumulative NAV", "Log Cumulative Return"],
    horizontal_spacing=0.15
)

# 첫 번째 그래프: 누적 NAV
fig.add_trace(
    go.Scatter(
        x=nav.index,
        y=nav.values,
        mode='lines',
        name=port_name,
        line=dict(color='#1f77b4', width=2)
    ),
    row=1, col=1
)

# 두 번째 그래프: 로그 누적 수익률
fig.add_trace(
    go.Scatter(
        x=log_cum.index,
        y=log_cum.values,
        mode='lines',
        name=port_name,
        line=dict(color='#1f77b4', width=2),
        showlegend=False
    ),
    row=1, col=2
)

# 레이아웃 설정
fig.update_layout(
    title=f"Portfolio Performance: {port_name}",
    height=500,
    width=1000,
    template="plotly_white",
    hovermode='x unified'
)

# 축 레이블 설정
fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Cumulative NAV", row=1, col=1)
fig.update_yaxes(title_text="Log Cumulative Return", row=1, col=2)

fig.show()

In [89]:
initial_NAV = 1.0
# 1) Return 정리 (inf 제거)
ret = factor_return.replace([np.inf, -np.inf], np.nan).dropna()
# 2) LS Trade 구성 (Q1 trade + Q5 trade)
q1_trade_df = pd.DataFrame({r['trade'].name: r['trade'] for r in res_q1})
q5_trade_df = pd.DataFrame({r['trade'].name: r['trade'] for r in res_q5})
ls_trade_df = pd.DataFrame(index=q1_trade_df.index)
for i in range(cap_cut):
    c = f"C({i+1}/{cap_cut})"
    # run_strategy_spread_qcut에서 trade portfolio_name이 아래 포맷으로 들어감
    ls_trade_df[f"{c}_LS(Q1-Q5)"] = q1_trade_df[f"{c}_Q(Q1)"] + q5_trade_df[f"{c}_Q(Q5)"]
factor_trade = ls_trade_df.mean(axis=1)
# 3) NAV 생성 (module.py 방식과 동일하게 initial_NAV로 시작)
nav = (1 + ret).cumprod() * initial_NAV
nav.iloc[0] = initial_NAV
# 4) Trade 정렬/정합 + NaN 처리
trade_aligned = factor_trade.reindex(ret.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
# 5) module.py에 넣을 portfolio DataFrame 만들기
portfolio = pd.DataFrame({
    "Return": ret,
    "NAV": nav,
    "Trade": trade_aligned
})
portfolio.index.name = "Date"
# 6) 성과지표 출력
metrics = performance_metrics(portfolio)
print("==== Long-Short (Q1-Q5) Performance Metrics ====")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

==== Long-Short (Q1-Q5) Performance Metrics ====
CAGR: 0.2214
Volatility (ann.): 0.1113
Sharpe Ratio: 1.8908
MDD: -0.0879
Average Turnover (quarterly): 0.4754


시총가중

In [90]:
from joblib import Parallel, delayed

cap_cut = 5
cap_buckets = list(range(cap_cut))  # 0~4

# Q1 leg
res_q1 = Parallel(n_jobs=-1)(
    delayed(run_strategy_spread_qcut)(
        wins_threshold[0], weight_method[1], cost[0],
        trading_threshold[0], Amihud_threshold[0],
        monthly_rets, illiq,
        c_temp=c, long_bucket='Q1', q_cut_local=5
    )
    for c in cap_buckets
)

# Q5 leg
res_q5 = Parallel(n_jobs=-1)(
    delayed(run_strategy_spread_qcut)(
        wins_threshold[0], weight_method[1], cost[0],
        trading_threshold[0], Amihud_threshold[0],
        monthly_rets, illiq,
        c_temp=c, long_bucket='Q5', q_cut_local=5
    )
    for c in cap_buckets
)

q1_df = pd.DataFrame({r['return'].name: r['return'] for r in res_q1})
q5_df = pd.DataFrame({r['return'].name: r['return'] for r in res_q5})

# c_temp별 롱숏: Q1 - Q5
ls_df = pd.DataFrame(index=q1_df.index)
for i in range(cap_cut):
    c = f"C({i+1}/{cap_cut})"
    ls_df[f"{c}_LS(Q1-Q5)"] = q1_df[f"{c}_Q(Q1)"] - q5_df[f"{c}_Q(Q5)"]

# 전체 평균 롱숏 팩터
factor_return = ls_df.mean(axis=1)
factor_return.name = "Factor_LS_Q1_minus_Q5"
factor_return.head()

2006-06-30    0.005347
2006-09-30    0.068591
2006-12-31    0.123254
2007-03-31    0.039279
2007-06-30    0.028793
Name: Factor_LS_Q1_minus_Q5, dtype: float64

In [91]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 포트폴리오/팩터 수익률 시계열 (직전 코드에서 만든 factor_return을 그대로 사용)
portfolio = factor_return
port_name = portfolio.name if portfolio.name is not None else "Factor_LS(Q1-Q5)"

# 데이터 준비
nav = (1 + portfolio).cumprod()                 # 누적 NAV
log_cum = np.log1p(portfolio).cumsum()        # 로그 누적 수익률

# 서브플롯 (가로 2개)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Cumulative NAV", "Log Cumulative Return"],
    horizontal_spacing=0.15
)

# 첫 번째 그래프: 누적 NAV
fig.add_trace(
    go.Scatter(
        x=nav.index,
        y=nav.values,
        mode='lines',
        name=port_name,
        line=dict(color='#1f77b4', width=2)
    ),
    row=1, col=1
)

# 두 번째 그래프: 로그 누적 수익률
fig.add_trace(
    go.Scatter(
        x=log_cum.index,
        y=log_cum.values,
        mode='lines',
        name=port_name,
        line=dict(color='#1f77b4', width=2),
        showlegend=False
    ),
    row=1, col=2
)

# 레이아웃 설정
fig.update_layout(
    title=f"Portfolio Performance: {port_name}",
    height=500,
    width=1000,
    template="plotly_white",
    hovermode='x unified'
)

# 축 레이블 설정
fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Cumulative NAV", row=1, col=1)
fig.update_yaxes(title_text="Log Cumulative Return", row=1, col=2)

fig.show()

In [92]:
initial_NAV = 1.0
# 1) Return 정리 (inf 제거)
ret = factor_return.replace([np.inf, -np.inf], np.nan).dropna()
# 2) LS Trade 구성 (Q1 trade + Q5 trade)
q1_trade_df = pd.DataFrame({r['trade'].name: r['trade'] for r in res_q1})
q5_trade_df = pd.DataFrame({r['trade'].name: r['trade'] for r in res_q5})
ls_trade_df = pd.DataFrame(index=q1_trade_df.index)
for i in range(cap_cut):
    c = f"C({i+1}/{cap_cut})"
    # run_strategy_spread_qcut에서 trade portfolio_name이 아래 포맷으로 들어감
    ls_trade_df[f"{c}_LS(Q1-Q5)"] = q1_trade_df[f"{c}_Q(Q1)"] + q5_trade_df[f"{c}_Q(Q5)"]
factor_trade = ls_trade_df.mean(axis=1)
# 3) NAV 생성 (module.py 방식과 동일하게 initial_NAV로 시작)
nav = (1 + ret).cumprod() * initial_NAV
nav.iloc[0] = initial_NAV
# 4) Trade 정렬/정합 + NaN 처리
trade_aligned = factor_trade.reindex(ret.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
# 5) module.py에 넣을 portfolio DataFrame 만들기
portfolio = pd.DataFrame({
    "Return": ret,
    "NAV": nav,
    "Trade": trade_aligned
})
portfolio.index.name = "Date"
# 6) 성과지표 출력
metrics = performance_metrics(portfolio)
print("==== Long-Short (Q1-Q5) Performance Metrics ====")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

==== Long-Short (Q1-Q5) Performance Metrics ====
CAGR: 0.2243
Volatility (ann.): 0.1064
Sharpe Ratio: 1.9970
MDD: -0.0930
Average Turnover (quarterly): 0.4913


In [93]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ===== 0) 데이터 준비 =====
# factor_return: 직전 셀에서 만든 롱숏 수익률 시계열 (Series)
portfolio = factor_return.copy()
portfolio.name = portfolio.name if portfolio.name is not None else "Factor_LS_Q1_minus_Q5"

# 분기 팩터 로드 (이미 있으면 생략 가능)
factors = pd.read_csv("../../00_input/factors quarterly.csv", index_col=0, parse_dates=True)

# 회귀용 DF 결합
df = pd.concat([portfolio.rename("Return"), factors], axis=1, join="inner")
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# ===== 1) 종속/독립변수 =====
# 종속변수: 초과수익률
y = df["Return"] - df["RF"]

# 4요인 (MKT, SMB, HML, MOM)
X = pd.DataFrame({
    "MKT": df["KOSPI"] - df["RF"],
    "SMB": df["SMB"],
    "HML": df["HML"],
    "MOM": df["MOM"],
}, index=df.index)
X = sm.add_constant(X)

# ===== 2) OLS (HAC Newey-West) =====
# 분기 데이터면 maxlags=4가 일반적(연 1년), 관성 강하면 8도 점검 가능
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})

print("=== OLS Regression (HAC) ===")
print(model.summary())

# ===== 3) 표 형태로 핵심 계수 점검 =====
coef_table = pd.DataFrame({
    "coef": model.params,
    "t": model.tvalues,
    "p": model.pvalues,
    "std_err": model.bse,
    "CI_low": model.conf_int()[0],
    "CI_high": model.conf_int()[1],
})
print("\n=== Coefficient Table ===")
print(coef_table.round(4))

# ===== 4) 진단표: 표본수, 설명력, 알파(연율) =====
alpha_q = model.params.get("const", np.nan)                # 분기 알파
alpha_ann = (1 + alpha_q)**4 - 1 if pd.notna(alpha_q) else np.nan

diag = pd.Series({
    "n_obs": int(model.nobs),
    "R2": model.rsquared,
    "Adj_R2": model.rsquared_adj,
    "alpha_quarterly": alpha_q,
    "alpha_annualized": alpha_ann,
    "AIC": model.aic,
    "BIC": model.bic,
})
print("\n=== Diagnostics ===")
print(diag.round(6))

# ===== 5) 다중공선성(VIF) =====
vif_df = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print("\n=== VIF ===")
print(vif_df.round(3))

=== OLS Regression (HAC) ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.025
Model:                            OLS   Adj. R-squared:                 -0.027
Method:                 Least Squares   F-statistic:                    0.6260
Date:                Wed, 01 Apr 2026   Prob (F-statistic):              0.645
Time:                        15:36:42   Log-Likelihood:                 122.85
No. Observations:                  79   AIC:                            -235.7
Df Residuals:                      74   BIC:                            -223.9
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0450  

In [94]:
def run_bottom_n_strategy(
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp, bottom_n_temp,
        monthly_rets, illiq,
        sort_mode='single',   # 'single', 'double'
        c_temp=None           # double sort 시 시가총액 분위수
):
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    # 포트폴리오 이름 생성
    if sort_mode == 'single':
        portfolio_name = (
            f"{weight_method_temp}_R{wins_threshold_temp}"
            f"_H{int(high_cost*10000)}L{int(low_cost*10000)}"
            f"_T{trading_threshold_temp}_A{Amihud_threshold_temp}"
            f"_Bottom{bottom_n_temp}"
        )
    elif sort_mode == 'double':
        portfolio_name = f"C({c_temp+1}/{cap_cut})_Bottom{bottom_n_temp}"
    else:
        raise ValueError("sort_mode는 'single' 또는 'double'만 가능합니다.")

    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    monthly_rets_win = monthly_rets.clip(
        lower=monthly_rets.quantile(wins_threshold_temp),
        upper=monthly_rets.quantile(1 - wins_threshold_temp),
        axis=1
    )

    prev_portfolio = None

    for i in range(len(month_ends) - 1):
        start_date, end_date = month_ends[i], month_ends[i + 1]

        # 시가총액 / 유동성 필터링
        if sort_mode == 'double':
            cap_series = mkt_cap.loc[start_date].dropna()
            cap_quantile = pd.qcut(cap_series, q=cap_cut, labels=False, duplicates='drop')
            filtered_index = cap_series[cap_quantile == c_temp].index
            series = trading_value_60.loc[start_date, filtered_index]
        else:
            series = trading_value_60.loc[start_date]

        series = series.dropna()
        threshold = series.quantile(trading_threshold_temp)
        filtered = series[series > threshold]

        factor_filtered = factor_df.loc[start_date, filtered.index].dropna().sort_values()

        # 하위 N개 종목 선택
        basket = factor_filtered.head(bottom_n_temp).index

        if len(basket) == 0:
            continue

        # Amihud high-cost 종목 선정
        illiq_startdate = illiq.loc[start_date].dropna()
        threshold = illiq_startdate.quantile(1 - Amihud_threshold_temp)
        illiq_top = illiq_startdate[illiq_startdate >= threshold].index

        # 이전 비중
        if prev_portfolio is None or prev_portfolio.sum() == 0:
            prev_weights = pd.Series(0.0, index=basket)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        # 목표 비중
        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1.0 / len(basket), index=basket)
        else:  # 'Cap'
            cap_seg = mkt_cap.loc[start_date, basket].dropna()
            cap_seg = cap_seg[cap_seg > 0]
            if len(cap_seg) == 0:
                continue
            basket = cap_seg.index
            target_weights = cap_seg / cap_seg.sum()

        # 거래 비용 계산
        all_index = target_weights.index.union(prev_weights.index)
        target_w = target_weights.reindex(all_index, fill_value=0)
        prev_w = prev_weights.reindex(all_index, fill_value=0)
        delta_w = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate = np.where(delta_w.index.isin(illiq_top), high_cost, low_cost)
        trade_cost = (trade_amounts * cost_rate).sum()

        NAV_after_cost = NAV - trade_cost

        # 다음 기 수익률 반영
        ret_seg = monthly_rets_win.loc[end_date, basket].dropna()
        if len(ret_seg) == 0:
            continue

        target_weights = target_weights.reindex(ret_seg.index).dropna()
        target_weights = target_weights / target_weights.sum()

        current_portfolio_value = target_weights * NAV_after_cost
        next_portfolio_value = current_portfolio_value * (ret_seg + 1)

        NAV_new = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV = NAV_new

        total_trade.loc[start_date] = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return {'return': portfolio_return, 'trade': total_trade}


### Bottom N 포트폴리오 실행 예시

`run_bottom_n_strategy(...)`를 사용해서 팩터 기준 하위 N개 종목에 투자하는 포트폴리오를 생성합니다.

In [95]:
bottom_n_temp = 10
wins_threshold_temp = wins_threshold[0]
weight_method_temp = weight_method[0]   # 'Cap'
cost_temp = cost[0]
trading_threshold_temp = trading_threshold[0]
Amihud_threshold_temp = Amihud_threshold[0]

result_bottom_n = run_bottom_n_strategy(
    wins_threshold_temp=wins_threshold_temp,
    weight_method_temp=weight_method_temp,
    cost_temp=cost_temp,
    trading_threshold_temp=trading_threshold_temp,
    Amihud_threshold_temp=Amihud_threshold_temp,
    bottom_n_temp=bottom_n_temp,
    monthly_rets=monthly_rets,
    illiq=illiq,
    sort_mode='single'
)

bottom_n_return = result_bottom_n['return'].replace([np.inf, -np.inf], np.nan).dropna()
bottom_n_trade = result_bottom_n['trade'].reindex(bottom_n_return.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)

bottom_n_nav = (1 + bottom_n_return).cumprod() * initial_NAV
if len(bottom_n_nav) > 0:
    bottom_n_nav.iloc[0] = initial_NAV

portfolio = pd.DataFrame({
    'Return': bottom_n_return,
    'NAV': bottom_n_nav,
    'Trade': bottom_n_trade,
})
portfolio.index.name = 'Date'

display(portfolio.head())

metrics = performance_metrics(portfolio)
print(f'=== Bottom {bottom_n_temp} Portfolio Performance Metrics ===')
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')


,Return,NAV,Trade
Date,,,
2006-06-30,-0.109510,1.000000,0.305741
2006-09-30,-0.033852,0.860344,0.752114
2006-12-31,0.129111,0.971424,0.473239
2007-03-31,0.060923,1.030606,1.137877
2007-06-30,0.170497,1.206322,0.211602


=== Bottom 10 Portfolio Performance Metrics ===
CAGR: 0.0521
Volatility (ann.): 0.2545
Sharpe Ratio: 0.3437
MDD: -0.5468
Average Turnover (quarterly): 0.6494


In [96]:
import plotly.express as px

# ==================================================
# 거래비용 반영 여부 비교
# 1) Cap-weighted Q(1/5)
# 2) Cap-weighted C(1/5)_Q(1/5)
# ==================================================

def build_portfolio_frame(return_series, trade_series, initial_nav=1.0):
    ret = return_series.replace([np.inf, -np.inf], np.nan).dropna()
    trade = trade_series.reindex(ret.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    nav = (1 + ret).cumprod() * initial_nav
    if len(nav) > 0:
        nav.iloc[0] = initial_nav
    portfolio = pd.DataFrame({
        'Return': ret,
        'NAV': nav,
        'Trade': trade,
    })
    portfolio.index.name = 'Date'
    return portfolio


def plot_cost_comparison(return_df, title):
    log_cum = np.log1p(return_df).cumsum()

    fig = px.line(
        log_cum,
        title=title,
        labels={'value': 'Log Cumulative Return', 'index': 'Date', 'variable': 'Portfolio'},
        template='plotly_white', width=900, height=600
    )
    fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)
    fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)
    fig.show()

    diff = return_df.iloc[:, 0] - return_df.iloc[:, 1]
    diff.name = 'With Cost - No Cost'
    diff_log_cum = np.log1p(diff).cumsum()

    fig_diff = px.line(
        diff_log_cum,
        title=f'{title} - Difference (With Cost minus No Cost)',
        labels={'value': 'Log Cumulative Return', 'index': 'Date'},
        template='plotly_white', width=900, height=400
    )
    fig_diff.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)
    fig_diff.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)
    fig_diff.show()


# -----------------------------
# 1. Single Sort: Q(1/5)
# -----------------------------
subset_q1_cost = select_columns(
    portfolio_df,
    {0: 'Cap', 1: 'R0.01', 2: '*', 3: 'T0.1', 4: 'A0.2', 5: 'Q(1/5)'}
)
subset_q1_trade = select_columns(
    trade_df,
    {0: 'Cap', 1: 'R0.01', 2: '*', 3: 'T0.1', 4: 'A0.2', 5: 'Q(1/5)'}
)

subset_q1_cost = subset_q1_cost.rename(columns={'H80L30': 'Q(1/5)_WithCost', 'H0L0': 'Q(1/5)_NoCost'})
subset_q1_trade = subset_q1_trade.rename(columns={'H80L30': 'Q(1/5)_WithCost', 'H0L0': 'Q(1/5)_NoCost'})

display(subset_q1_cost.head())
plot_cost_comparison(
    subset_q1_cost[['Q(1/5)_WithCost', 'Q(1/5)_NoCost']],
    'Cap-weighted Q(1/5): Cost vs No Cost'
)

portfolio_q1_with_cost = build_portfolio_frame(
    subset_q1_cost['Q(1/5)_WithCost'],
    subset_q1_trade['Q(1/5)_WithCost'],
    initial_NAV,
)
portfolio_q1_no_cost = build_portfolio_frame(
    subset_q1_cost['Q(1/5)_NoCost'],
    subset_q1_trade['Q(1/5)_NoCost'],
    initial_NAV,
)

metrics_q1 = pd.DataFrame({
    'Q(1/5)_WithCost': performance_metrics(portfolio_q1_with_cost),
    'Q(1/5)_NoCost': performance_metrics(portfolio_q1_no_cost),
}).T
display(metrics_q1)


# -----------------------------
# 2. Double Sort: C(1/5)_Q(1/5)
# -----------------------------
double_with_cost = run_strategy(
    wins_threshold_temp=wins_threshold[0],
    weight_method_temp='Cap',
    cost_temp=cost[0],
    trading_threshold_temp=trading_threshold[0],
    Amihud_threshold_temp=Amihud_threshold[0],
    n_temp=0,
    monthly_rets=monthly_rets,
    illiq=illiq,
    sort_mode='double',
    c_temp=0,
)

double_no_cost = run_strategy(
    wins_threshold_temp=wins_threshold[0],
    weight_method_temp='Cap',
    cost_temp=cost[1],
    trading_threshold_temp=trading_threshold[0],
    Amihud_threshold_temp=Amihud_threshold[0],
    n_temp=0,
    monthly_rets=monthly_rets,
    illiq=illiq,
    sort_mode='double',
    c_temp=0,
)

double_cost_df = pd.DataFrame({
    'C(1/5)_Q(1/5)_WithCost': double_with_cost['return'],
    'C(1/5)_Q(1/5)_NoCost': double_no_cost['return'],
})
double_trade_df = pd.DataFrame({
    'C(1/5)_Q(1/5)_WithCost': double_with_cost['trade'],
    'C(1/5)_Q(1/5)_NoCost': double_no_cost['trade'],
})

display(double_cost_df.head())
plot_cost_comparison(
    double_cost_df[['C(1/5)_Q(1/5)_WithCost', 'C(1/5)_Q(1/5)_NoCost']],
    'Cap-weighted C(1/5)_Q(1/5): Cost vs No Cost'
)

portfolio_double_with_cost = build_portfolio_frame(
    double_cost_df['C(1/5)_Q(1/5)_WithCost'],
    double_trade_df['C(1/5)_Q(1/5)_WithCost'],
    initial_NAV,
)
portfolio_double_no_cost = build_portfolio_frame(
    double_cost_df['C(1/5)_Q(1/5)_NoCost'],
    double_trade_df['C(1/5)_Q(1/5)_NoCost'],
    initial_NAV,
)

metrics_double = pd.DataFrame({
    'C(1/5)_Q(1/5)_WithCost': performance_metrics(portfolio_double_with_cost),
    'C(1/5)_Q(1/5)_NoCost': performance_metrics(portfolio_double_no_cost),
}).T
display(metrics_double)


,Q(1/5)_WithCost,Q(1/5)_NoCost
2006-06-30,-0.027738,-0.024783
2006-09-30,0.052786,0.052941
2006-12-31,0.045113,0.045614
2007-03-31,-0.034664,-0.034314
2007-06-30,0.113622,0.114352


,CAGR,Volatility (ann.),Sharpe Ratio,MDD,Average Turnover (quarterly)
Q(1/5)_WithCost,0.118792,0.185046,0.710661,-0.371496,0.161256
Q(1/5)_NoCost,0.121201,0.185130,0.721490,-0.370352,0.161256


,C(1/5)_Q(1/5)_WithCost,C(1/5)_Q(1/5)_NoCost
2006-06-30,-0.052934,-0.048336
2006-09-30,0.079802,0.083859
2006-12-31,0.099562,0.103366
2007-03-31,0.364034,0.368744
2007-06-30,0.266392,0.272687


,CAGR,Volatility (ann.),Sharpe Ratio,MDD,Average Turnover (quarterly)
C(1/5)_Q(1/5)_WithCost,0.277193,0.257320,1.112068,-0.417577,0.786784
C(1/5)_Q(1/5)_NoCost,0.297457,0.258928,1.169929,-0.407399,0.786784


In [97]:
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")

# double sort with cost / no cost 수익률 저장
double_cost_df.to_csv(
    out_dir / "double_sort_C1Q1_return.csv",
    index_label="Date",
    encoding="utf-8-sig"
)
double_trade_df.to_csv(
    out_dir / "double_sort_C1Q1_trade.csv",
    index_label="Date",
    encoding="utf-8-sig"
)
metrics_double.to_csv(
    out_dir / "double_sort_C1Q1_metrics.csv",
    encoding="utf-8-sig"
)
print("✓ 저장 완료")
print(f"  - {out_dir / 'double_sort_C1Q1_return.csv'}")
print(f"  - {out_dir / 'double_sort_C1Q1_trade.csv'}")

✓ 저장 완료
  - 01_output\01_quarterly\double_sort_C1Q1_return.csv
  - 01_output\01_quarterly\double_sort_C1Q1_trade.csv


In [98]:
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")

# ── 수익률 원본 ──────────────────────────────────────────────────────────
double_cost_df.to_csv(
    out_dir / "double_sort_C1Q1_return.csv",
    index_label="Date", encoding="utf-8-sig"
)

# ── 누적수익률: (1 + r).cumprod() - 1 ────────────────────────────────────
cum_ret = (1 + double_cost_df).cumprod() - 1
cum_ret.to_csv(
    out_dir / "double_sort_C1Q1_cumulative_return.csv",
    index_label="Date", encoding="utf-8-sig"
)

# ── 로그 누적수익률: log(1 + r)의 누적합 ──────────────────────────────────
log_cum_ret = np.log1p(double_cost_df).cumsum()
log_cum_ret.to_csv(
    out_dir / "double_sort_C1Q1_log_cumulative_return.csv",
    index_label="Date", encoding="utf-8-sig"
)

print("✓ 저장 완료")
print(f"  - double_sort_C1Q1_return.csv")
print(f"  - double_sort_C1Q1_cumulative_return.csv")
print(f"  - double_sort_C1Q1_log_cumulative_return.csv")

✓ 저장 완료
  - double_sort_C1Q1_return.csv
  - double_sort_C1Q1_cumulative_return.csv
  - double_sort_C1Q1_log_cumulative_return.csv


In [99]:
import statsmodels.api as sm
from pathlib import Path

out_dir = Path("./01_output/01_quarterly")

# ── 팩터 데이터 로드 ────────────────────────────────────────────────────────
factors = pd.read_csv(
    '../../11_메인세션 1주차 (26.03.13)/01_Advanced Backtesting/1. Factor Construction/output/factors quarterly.csv',
    index_col='Date', parse_dates=True
)

# ── 회귀 함수 (CAPM / FF3 / FF4 선택 가능) ──────────────────────────────────
def run_factor_regression(portfolio_ret, factors_df, model='FF4'):
    common_idx  = portfolio_ret.index.intersection(factors_df.index)
    ret         = portfolio_ret.loc[common_idx]
    fac         = factors_df.loc[common_idx]

    # 초과수익률
    excess_ret  = ret - fac['RF']

    # KOSPI 초과수익률
    mkt_excess  = fac['KOSPI'] - fac['RF']

    if model == 'CAPM':
        X = pd.DataFrame({'MKT': mkt_excess})
    elif model == 'FF3':
        X = pd.DataFrame({'MKT': mkt_excess, 'SMB': fac['SMB'], 'HML': fac['HML']})
    elif model == 'FF4':
        X = pd.DataFrame({'MKT': mkt_excess, 'SMB': fac['SMB'], 'HML': fac['HML'], 'MOM': fac['MOM']})

    X   = sm.add_constant(X)
    fit = sm.OLS(excess_ret, X, missing='drop').fit(cov_type='HC3')  # 이분산 강건 표준오차
    return fit

# ── 회귀 실행 ────────────────────────────────────────────────────────────────
portfolios = {
    'WithCost': double_cost_df['C(1/5)_Q(1/5)_WithCost'],
    'NoCost':   double_cost_df['C(1/5)_Q(1/5)_NoCost'],
}

reg_results = {}
for name, ret_series in portfolios.items():
    print("=" * 60)
    print(f"C(1/5)_Q(1/5) {name} — FF4 회귀")
    print("=" * 60)
    fit = run_factor_regression(ret_series, factors, model='FF4')
    print(fit.summary())
    reg_results[name] = fit

# ── 결과 요약 표 ─────────────────────────────────────────────────────────────
rows = {}
for name, fit in reg_results.items():
    rows[name] = {
        'Alpha (분기)':     fit.params['const'],
        'Alpha (연율화)':   fit.params['const'] * 4,       # 분기 → 연율화
        'Alpha t-stat':    fit.tvalues['const'],
        'Alpha p-value':   fit.pvalues['const'],
        'Beta_MKT':        fit.params['MKT'],
        'Beta_SMB':        fit.params['SMB'],
        'Beta_HML':        fit.params['HML'],
        'Beta_MOM':        fit.params['MOM'],
        'R²':              fit.rsquared,
        'Adj. R²':         fit.rsquared_adj,
        'N':               int(fit.nobs),
    }

reg_summary = pd.DataFrame(rows).T.round(4)
display(reg_summary)

# ── 저장 ─────────────────────────────────────────────────────────────────────
reg_summary.to_csv(
    out_dir / "double_sort_C1Q1_factor_regression.csv",
    encoding="utf-8-sig"
)
print("\n✓ 회귀 결과 저장 완료")

C(1/5)_Q(1/5) WithCost — FF4 회귀
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.849
Model:                            OLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                     187.1
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           6.94e-38
Time:                        15:36:48   Log-Likelihood:                 124.56
No. Observations:                  79   AIC:                            -239.1
Df Residuals:                      74   BIC:                            -227.3
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.055

,Alpha (분기),Alpha (연율화),Alpha t-stat,Alpha p-value,Beta_MKT,Beta_SMB,Beta_HML,Beta_MOM,R²,Adj. R²,N
WithCost,0.0552,0.2208,10.1270,0.0,0.8767,1.3599,0.0666,-0.1849,0.8487,0.8405,79.0
NoCost,0.0593,0.2373,10.8288,0.0,0.8823,1.3690,0.0671,-0.1842,0.8494,0.8412,79.0



✓ 회귀 결과 저장 완료


In [100]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

# ── 성과 지표 함수 (일관성 있게 전체 기간 사용) ────────────────────────────
def calc_performance(ret: pd.Series, factors_df: pd.DataFrame,
                     periods_per_year: int = 4) -> dict:
    r   = ret.dropna()
    n   = len(r)
    fac = factors_df.reindex(r.index).dropna()
    r_  = r.reindex(fac.index)  # 팩터와 날짜 맞춤

    # 1. CAGR
    nav        = (1 + r).cumprod()
    years      = n / periods_per_year
    cagr       = nav.iloc[-1] ** (1 / years) - 1

    # 2. 연율화 변동성 (ddof=1)
    vol_ann    = r.std(ddof=1) * np.sqrt(periods_per_year)

    # 3. 샤프비율 (RF=0 가정)
    sharpe     = (r.mean() * periods_per_year) / vol_ann if vol_ann != 0 else np.nan

    # 4. MDD
    cummax     = nav.cummax()
    mdd        = (nav / cummax - 1).min()

    # 5. FF4 알파 (KOSPI − RF 사용)
    excess_ret = r_ - fac['RF']
    mkt_excess = fac['KOSPI'] - fac['RF']
    X = sm.add_constant(pd.DataFrame({
        'MKT': mkt_excess,
        'SMB': fac['SMB'],
        'HML': fac['HML'],
        'MOM': fac['MOM'],
    }))
    fit          = sm.OLS(excess_ret, X, missing='drop').fit()
    alpha_q      = fit.params['const']
    alpha_ann    = alpha_q * periods_per_year
    alpha_tstat  = fit.tvalues['const']
    alpha_pvalue = fit.pvalues['const']

    return {
        'N':                  n,
        'CAGR':               round(cagr,        4),
        'Volatility (ann.)':  round(vol_ann,      4),
        'Sharpe (RF=0)':      round(sharpe,       4),
        'MDD':                round(mdd,          4),
        'Alpha (분기)':        round(alpha_q,      6),
        'Alpha (연율화)':      round(alpha_ann,    4),
        'Alpha t-stat':       round(alpha_tstat,  4),
        'Alpha p-value':      round(alpha_pvalue, 4),
        'R²':                 round(fit.rsquared, 4),
    }

# ── 계산 ───────────────────────────────────────────────────────────────────
metrics = {
    'C(1/5)_Q(1/5)_WithCost': calc_performance(
        double_cost_df['C(1/5)_Q(1/5)_WithCost'], factors),
    'C(1/5)_Q(1/5)_NoCost':   calc_performance(
        double_cost_df['C(1/5)_Q(1/5)_NoCost'],   factors),
}

result_df = pd.DataFrame(metrics).T
display(result_df)

# ── 저장 ───────────────────────────────────────────────────────────────────
out_dir = Path("./01_output/01_quarterly")
result_df.to_csv(out_dir / "double_sort_C1Q1_performance.csv", encoding="utf-8-sig")
print("✓ 저장 완료")

,N,CAGR,Volatility (ann.),Sharpe (RF=0),MDD,Alpha (분기),Alpha (연율화),Alpha t-stat,Alpha p-value,R²
C(1/5)_Q(1/5)_WithCost,79.0,0.2732,0.2572,1.0881,-0.4176,0.055206,0.2208,8.7757,0.0,0.8487
C(1/5)_Q(1/5)_NoCost,79.0,0.2932,0.2588,1.1464,-0.4074,0.059324,0.2373,9.3933,0.0,0.8494


✓ 저장 완료


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PERIODS_PER_YEAR = 4
START = '2006-03-31'
END   = '2025-12-31'

# ── KOSPI 지수 로드 및 분기 수익률 계산 ─────────────────────────────────────
kospi_raw = pd.read_csv(
    '../../00_input/KOSPI_index.csv',
    index_col='Date', parse_dates=True
).squeeze()

# 분기말 종가 → 분기 수익률 (portfolio_df와 동일 방식)
kospi_q   = kospi_raw.resample('QE').last().dropna()
kospi_ret = kospi_q.pct_change(fill_method=None).dropna()

# 백테스트 기간과 동일하게 필터링
kospi_ret = kospi_ret.loc[START:END]

# ── 성과 지표 계산 ──────────────────────────────────────────────────────────
def calc_metrics(ret: pd.Series, name: str = "") -> dict:
    r     = ret.dropna()
    n     = len(r)
    nav   = (1 + r).cumprod()
    years = n / PERIODS_PER_YEAR
    cagr  = nav.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol_ann = r.std(ddof=1) * np.sqrt(PERIODS_PER_YEAR)
    sharpe  = (r.mean() * PERIODS_PER_YEAR) / vol_ann if vol_ann != 0 else np.nan
    mdd     = (nav / nav.cummax() - 1).min()

    return {
        "기간":          f"{r.index[0].date()} ~ {r.index[-1].date()}",
        "N (분기)":      n,
        "CAGR":          round(cagr,    4),
        "Volatility":    round(vol_ann, 4),
        "Sharpe (RF=0)": round(sharpe,  4),
        "MDD":           round(mdd,     4),
    }

kospi_metrics = pd.DataFrame({'KOSPI': calc_metrics(kospi_ret)}).T
print("[ KOSPI 벤치마크 성과 ]")
display(kospi_metrics)

# ── 포트폴리오와 나란히 비교 ─────────────────────────────────────────────────
# (portfolio_df, double_cost_df가 이미 정의된 상태라고 가정)
comparison = pd.concat([
    pd.DataFrame({'KOSPI': calc_metrics(kospi_ret)}).T,
    pd.DataFrame({'C(1/5)_Q(1/5)_WithCost': calc_metrics(double_cost_df['C(1/5)_Q(1/5)_WithCost'])}).T,
    pd.DataFrame({'C(1/5)_Q(1/5)_NoCost':   calc_metrics(double_cost_df['C(1/5)_Q(1/5)_NoCost'])}).T,
])
print("\n[ 벤치마크 vs 포트폴리오 비교 ]")
display(comparison)

# ── 저장 ───────────────────────────────────────────────────────────────────
out_dir = Path("./01_output/01_quarterly")
comparison.to_csv(out_dir / "benchmark_vs_portfolio_metrics.csv", encoding="utf-8-sig")
print("\n✓ 저장 완료")

[ KOSPI 벤치마크 성과 ]


,기간,N (분기),CAGR,Volatility,Sharpe (RF=0),MDD
KOSPI,2006-03-31 ~ 2025-12-31,80,0.0574,0.185,0.3942,-0.4223



[ 벤치마크 vs 포트폴리오 비교 ]


,기간,N (분기),CAGR,Volatility,Sharpe (RF=0),MDD
KOSPI,2006-03-31 ~ 2025-12-31,80,0.0574,0.185,0.3942,-0.4223
C(1/5)_Q(1/5)_WithCost,2006-06-30 ~ 2025-12-31,79,0.2732,0.2572,1.0881,-0.4176
C(1/5)_Q(1/5)_NoCost,2006-06-30 ~ 2025-12-31,79,0.2932,0.2588,1.1464,-0.4074



✓ 저장 완료


: 